In [ ]:
# ============================================================
# SKETCHBYTE — CELL 2
# GOOGLE DRIVE + REFERENCE VOICE + QWEN3-4B FORMATTER
# ============================================================

import os
import gc
import torch
import soundfile as sf

if "_sb_error" not in globals():
    def _sb_error(title, detail, hint=None):
        lines = [
            "=" * 70,
            "❌ SKETCHBYTE ERROR",
            "=" * 70,
            "",
            str(title),
        ]
        detail_text = str(detail).strip()
        if detail_text:
            lines += ["", detail_text]
        if hint:
            lines += ["", f"💡 HINT: {hint}"]
        return RuntimeError("\n".join(lines))

print("=" * 70)
print("SKETCHBYTE — CELL 2")
print("GOOGLE DRIVE + REFERENCE VOICE + QWEN3-4B")
print("=" * 70)


# ============================================================
# 1. CHECK CELL 1
# ============================================================

print("\n[1/5] Checking Cell 1 configuration...")
print("-" * 70)

required_variables = [
    "FORMATTER_MODEL",
    "TTS_MODEL",
    "REFERENCE_AUDIO",
    "REFERENCE_TEXT",
    "OUTPUT_FILE"
]

missing = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing:
    raise RuntimeError(
        "\n❌ Cell 1 has not been completed.\n\n"
        "Missing variables:\n"
        + "\n".join(f"• {x}" for x in missing)
        + "\n\nRun Cell 1 first."
    )

print("✅ Cell 1 configuration found.")


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

print("\n[2/5] Connecting to Google Drive...")
print("-" * 70)

try:
    from google.colab import drive
    drive.mount(
        "/content/drive",
        force_remount=False
    )
except Exception as e:
    raise _sb_error(
        "GOOGLE DRIVE MOUNT FAILED",
        str(e),
        "This notebook runs only in Google Colab. Reconnect the runtime, allow Drive access, then re-run Cell 2.",
    )

print("✅ Google Drive connected.")


# ============================================================
# 3. LOCATE PERMANENT REFERENCE VOICE
# ============================================================

print("\n[3/5] Locating permanent SketchByte voice...")
print("-" * 70)

SKETCHBYTE_FOLDER = (
    "/content/drive/MyDrive/SketchByte"
)

VOICE_FOLDER = os.path.join(
    SKETCHBYTE_FOLDER,
    "Voice"
)

REFERENCE_PATH = os.path.join(
    VOICE_FOLDER,
    REFERENCE_AUDIO
)

# If REFERENCE_AUDIO already contains a full path,
# use it directly.
if os.path.isabs(REFERENCE_AUDIO):
    REFERENCE_PATH = REFERENCE_AUDIO


if not os.path.isdir(SKETCHBYTE_FOLDER):
    raise FileNotFoundError(
        "\n❌ SketchByte folder not found.\n\n"
        "Expected:\n"
        "My Drive/SketchByte/"
    )


if not os.path.isdir(VOICE_FOLDER):
    raise FileNotFoundError(
        "\n❌ Voice folder not found.\n\n"
        "Expected:\n"
        "My Drive/SketchByte/Voice/"
    )


if not os.path.isfile(REFERENCE_PATH):

    available = os.listdir(VOICE_FOLDER)

    raise FileNotFoundError(
        "\n"
        "====================================================\n"
        "❌ REFERENCE AUDIO NOT FOUND\n"
        "====================================================\n\n"
        f"Expected:\n{REFERENCE_PATH}\n\n"
        "Files currently in Voice folder:\n"
        + (
            "\n".join(
                f"• {x}" for x in available
            )
            if available
            else "• Folder is empty"
        )
    )


REFERENCE_AUDIO = REFERENCE_PATH

print("✅ Voice folder found.")
print(f"🎧 Reference:\n   {REFERENCE_AUDIO}")


# ============================================================
# 4. NORMALIZE + VALIDATE AUDIO + TRANSCRIPT
# ============================================================

print("\n[4/5] Preparing reference voice...")
print("-" * 70)

# ------------------------------------------------------------
# 4.1 NORMALIZE REFERENCE (MP3/OGG/etc -> canonical 24 kHz mono WAV)
# qwen-tts reads local files via librosa; converting once to a
# canonical WAV removes decode and sample-rate surprises.
# This MUST run BEFORE validation: soundfile only decodes MP3 on
# specific libsndfile builds, while librosa decodes any format.
# ------------------------------------------------------------

if not REFERENCE_AUDIO.lower().endswith(".wav"):

    import librosa

    print("\n🔧 Format is not WAV. Converting to canonical 24 kHz mono WAV...")

    try:

        wav_path = os.path.splitext(REFERENCE_AUDIO)[0] + "_ref_24k.wav"

        y, sr = librosa.load(
            REFERENCE_AUDIO,
            sr=24000,
            mono=True
        )

        sf.write(wav_path, y, sr)

        REFERENCE_AUDIO = wav_path

        print(f"✅ Canonical reference: {REFERENCE_AUDIO}")

    except Exception as e:
        raise RuntimeError(
            "\n❌ Reference audio could not be decoded/converted.\n\n"
            f"{e}\n\n"
            "Check that the file is a standard MP3, WAV, FLAC, M4A, or OGG."
        )

else:
    print("✅ Format is already WAV. No conversion needed.")


# ------------------------------------------------------------
# 4.2 VALIDATE THE (NOW CANONICAL) WAV
# ------------------------------------------------------------

print("\n✅ Validating reference voice...")

try:

    info = sf.info(
        REFERENCE_AUDIO
    )

    if info.duration <= 0:
        raise ValueError(
            "Audio duration is zero."
        )

    print(
        f"✅ Duration: "
        f"{info.duration:.2f} seconds"
    )

    print(
        f"✅ Sample rate: "
        f"{info.samplerate:,} Hz"
    )

    print(
        f"✅ Channels: "
        f"{info.channels}"
    )

    if info.duration < 3:
        print(
            f"⚠️  Warning: reference is only {info.duration:.1f}s. "
            "Voice cloning works best with 3+ seconds of clean speech."
        )

except Exception as e:

    raise RuntimeError(
        "\n❌ Reference audio could not be read.\n\n"
        f"{e}"
    )


# ------------------------------------------------------------
# 4.3 TRANSCRIPT
# ------------------------------------------------------------

if not REFERENCE_TEXT.strip():
    raise ValueError(
        "\n❌ REFERENCE_TEXT is empty.\n"
        "Check Cell 1."
    )

print(
    f"✅ Reference transcript: "
    f"{len(REFERENCE_TEXT.split())} words"
)


# ------------------------------------------------------------
# 4.4 PERSIST REFERENCE STATE (SURVIVES RUNTIME RESTARTS)
# ------------------------------------------------------------

import json

STATE_FILE = "/content/sketchbyte_state.json"

try:
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(
            {
                "REFERENCE_AUDIO": REFERENCE_AUDIO,
                "REFERENCE_TEXT": REFERENCE_TEXT,
                "OUTPUT_FILE": OUTPUT_FILE,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )
except Exception as e:
    raise _sb_error(
        "REFERENCE STATE SAVE FAILED",
        str(e),
        "Free up /content disk space, then re-run Cell 2.",
    )

print(f"✅ Reference state saved: {STATE_FILE}")


# ============================================================
# 5. LOAD QWEN3-4B FORMATTER
# ============================================================

print("\n[5/5] Loading Qwen3-4B-Instruct-2507...")
print("-" * 70)

print(
    "\n⏳ Loading model..."
)
print(
    "This may take a while on the first run."
)

try:

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM
    )

    # --------------------------------------------------------
    # TOKENIZER
    # --------------------------------------------------------

    print("\n📥 Loading tokenizer...")

    formatter_tokenizer = (
        AutoTokenizer.from_pretrained(
            FORMATTER_MODEL,
            trust_remote_code=True
        )
    )

    print("✅ Tokenizer loaded.")


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    print("\n📥 Loading Qwen3-4B...")

    formatter_model = (
        AutoModelForCausalLM.from_pretrained(
            FORMATTER_MODEL,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
    )

    formatter_model.eval()

    print("✅ Qwen3-4B loaded successfully.")


except Exception as e:

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    raise RuntimeError(
        "\n"
        "====================================================\n"
        "❌ QWEN3-4B LOADING FAILED\n"
        "====================================================\n\n"
        f"Error:\n{e}\n"
    )


# ============================================================
# MEMORY STATUS
# ============================================================

print("\n" + "=" * 70)
print("SYSTEM STATUS")
print("=" * 70)

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    allocated = (
        torch.cuda.memory_allocated()
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )

    print(f"\nGPU       : {gpu_name}")
    print(f"VRAM used : {allocated:.2f} GB")
    print(f"VRAM cache: {reserved:.2f} GB")

else:

    print("\n❌ No GPU detected.")


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("🟢 CELL 2 COMPLETE")
print("=" * 70)

print("\nGoogle Drive       : ✅")
print("Reference voice    : ✅")
print("Reference text     : ✅")
print("Qwen3-4B formatter : ✅")

print("\n🎧 Permanent voice:")
print(REFERENCE_AUDIO)

print("\n➡️ NEXT: RUN CELL 3")

print("=" * 70)